In [9]:
import pandas as pd
from datetime import datetime, timezone, timedelta
import csv
import os
import requests
import json
import time
from collections import deque

In [ ]:
# pushpull reddit search api로 순회 설정
# Reddit 전체는 불가능하다!! 반드시 subreddit 단위로 접근
SUBREDDITS = [
    "leagueoflegends",
    "Jungle_Mains",
    "supportlol",
    "summonerschool"
]

start_date = datetime(2025, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
end_date   = datetime(2026, 1, 31, 23, 59, 59, tzinfo=timezone.utc) # 테스트용
#end_date   = datetime(2026, 2, 4, 23, 59, 59, tzinfo=timezone.utc) 

start_ts = int(start_date.timestamp())
end_ts   = int(end_date.timestamp())

SUBREDDITS = [
    "leagueoflegends",
    "summonerschool",
    "CompetitiveLoL"
]

OUTPUT_FILE = "lol_reddit_full_250101_260204.csv"

BASE_SUBMISSION_URL = "https://api.pullpush.io/reddit/search/submission/"
BASE_COMMENT_URL    = "https://api.pullpush.io/reddit/search/comment/"

BATCH_SIZE = 100
REQUEST_DELAY = 1.0


In [4]:
# 중복 방지
saved_ids = set()

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            saved_ids.add(row["post_id"])

print(f"이미 저장된 게시글 수: {len(saved_ids)}")

이미 저장된 게시글 수: 98


In [5]:
# csv 초기화
# 게시글 고유 id, 서브레딧 이름, 작성한 시간, 점수, 댓글 개수, 텍스트 full(제목, 본문, 댓글) <- 전처리 필요 
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "post_id",
            "created_utc",
            "subreddit",
            "title",
            "selftext",
            "comment_id",
            "comment_body"
        ])

In [7]:
# 하나의 게시글(post_id)에 달린 모든 댓글을 수집하는 함수
# PullPush는 한 번에 최대 100개만 반환하니까 파라미터를 갱신하면서 반복 호출 필요
def fetch_comments(post_id):
    comments = []
    params = {
        "link_id": f"t3_{post_id}",
        "size": 500
    }

    try:
        res = requests.get(BASE_COMMENT_URL, params=params)
        data = res.json().get("data", [])
        for c in data:
            comments.append({
                "comment_id": c.get("id"),
                "comment_body": c.get("body", "")
            })
    except:
        pass

    return comments

In [ ]:
# 메인 실행
recent_times = deque(maxlen=50)
total_processed = 0

for subreddit in SUBREDDITS:

    print(f"\n===== r/{subreddit} 수집 시작 =====")

    current_after = start_ts

    while current_after < end_ts:

        params = {
            "subreddit": subreddit,
            "after": current_after,
            "before": end_ts,
            "size": BATCH_SIZE,
            "sort": "asc",
            "sort_type": "created_utc"
        }

        try:
            res = requests.get(BASE_SUBMISSION_URL, params=params)
            data = res.json().get("data", [])
        except:
            print("요청 실패. 5초 후 재시도")
            time.sleep(5)
            continue

        if not data:
            break

        for post in data:

            start_item_time = time.time()

            post_id = post.get("id")
            created_utc = post.get("created_utc")

            if post_id in saved_ids:
                continue

            # UTC datetime 문자열 변환
            created_datetime_str = datetime.fromtimestamp(
                created_utc,
                timezone.utc
            ).strftime("%Y-%m-%d %H:%M:%S")

            comments = fetch_comments(post_id)

            with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)

                if not comments:
                    writer.writerow([
                        post_id,
                        created_utc,
                        created_datetime_str,
                        subreddit,
                        post.get("title"),
                        post.get("selftext"),
                        "",
                        ""
                    ])
                else:
                    for c in comments:
                        writer.writerow([
                            post_id,
                            created_utc,
                            created_datetime_str,
                            subreddit,
                            post.get("title"),
                            post.get("selftext"),
                            c["comment_id"],
                            c["comment_body"]
                        ])

            saved_ids.add(post_id)
            total_processed += 1

            # ============================
            # 진행률 + ETA 계산
            # ============================

            elapsed_item = time.time() - start_item_time
            recent_times.append(elapsed_item)

            avg_time = sum(recent_times) / len(recent_times)

            progress_ratio = (created_utc - start_ts) / (END_TS - start_ts)
            progress_ratio = max(0, min(progress_ratio, 1))

            remaining_ratio = 1 - progress_ratio

            remaining_time_est = (
                remaining_ratio * (total_processed * avg_time) / progress_ratio
                if progress_ratio > 0 else 0
            )

            current_time_str = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

            print(
                f"[{current_time_str} UTC] "
                f"저장 중 ID: {post_id} | "
                f"게시글 UTC 날짜: {created_datetime_str} | "
                f"진행률: {progress_ratio*100:.2f}% | "
                f"총 처리: {total_processed}개 | "
                f"평균: {avg_time:.2f}초/개 | "
                f"예상 남은 시간: {remaining_time_est/60:.1f}분"
            )

            time.sleep(REQUEST_DELAY)

        current_after = data[-1]["created_utc"]
        
print("\n====================전체 수집 완료====================")


===== r/leagueoflegends 수집 시작 =====


NameError: name 'END_TS' is not defined

In [ ]:
# 필요하다면 진행 정글, 서폿 관련 용어 필터링 작업

df = pd.read_csv("lol_reddit_full_2025_2026.csv")

keywords = ["jungle", "jungler", "support", "roam", "gank"]
df_filtered = df[df["full_text"].str.contains("|".join(keywords), case=False)]
df_filtered.to_csv("lol_jungle_support_filtered.csv", index=False)